# AI Model Experiment & Evaluation

Notebook ini membandingkan dua pendekatan untuk sentiment analysis ulasan pelanggan:

1. **Classic ML**: TF-IDF dan Logistic Regression yang dilatih menggunakan training set.
2. **Gemini API**: zero-shot/few-shot prompting pada test set yang sama.

Split data dibuat sekali dengan `random_state=42` dan `stratify`, kemudian dipakai oleh kedua pendekatan agar perbandingan adil.

## 1. Import dan konfigurasi

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebook':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from run_experiment import (
    DATA_PATH,
    DOCUMENTATION_DIR,
    LABELS,
    RESULTS_DIR,
    build_gemini_prompt,
    calculate_metrics,
    load_dataset,
    predict_with_gemini,
    save_confusion_matrices,
    split_dataset,
    train_classic_model,
)

pd.set_option('display.max_colwidth', 120)


## 2. Load dataset dan membuat test set

Dataset asli hanya dibaca. Kolom `review_text` dipakai sebagai fitur dan `sentiment` sebagai target. Kolom produk tidak dipakai agar model fokus pada isi ulasan.

In [ ]:
data = load_dataset(DATA_PATH)
train_data, test_data = split_dataset(data)

print(f'Jumlah data: {len(data)}')
print(f'Training set: {len(train_data)} baris')
print(f'Test set: {len(test_data)} baris')
print('Distribusi label:')
display(data['sentiment'].value_counts().rename_axis('label').to_frame('jumlah'))

display(test_data.head())


## 3. Pendekatan 1 — Classic ML

TF-IDF mengubah teks menjadi fitur numerik. Logistic Regression kemudian belajar membedakan label `positif` dan `negatif` dari training set.

In [ ]:
classic_model = train_classic_model(train_data)
classic_predictions = classic_model.predict(test_data['review_text']).tolist()
classic_metrics, classic_matrix, classic_report = calculate_metrics(
    test_data['sentiment'], classic_predictions
)

print('Metrik Classic ML:')
display(pd.DataFrame([classic_metrics], index=['Classic ML']).round(4))
print('Classification report:')
display(classic_report.round(4))
print('Confusion matrix:')
display(classic_matrix)


## 4. Pendekatan 2 — Gemini API

Prompt menggunakan dua contoh singkat (few-shot) dan memaksa output satu kata agar mudah diparse. Untuk Gemini 3.x, `thinking_level='low'` dipilih karena klasifikasi ini sederhana dan tidak membutuhkan reasoning panjang.

Cell ini otomatis menjalankan API jika `GEMINI_API_KEY` tersedia. Jika belum ada key, notebook tetap dapat dijalankan dan memberi instruksi tanpa membuat metrik Gemini palsu.

In [ ]:
print(build_gemini_prompt('Packing rapi dan produknya sesuai deskripsi.'))

api_key = os.getenv('GEMINI_API_KEY')
run_gemini = os.getenv('RUN_GEMINI', 'auto').lower()
should_run_gemini = run_gemini == 'true' or (run_gemini == 'auto' and bool(api_key))

if should_run_gemini and api_key:
    gemini_predictions, gemini_raw_responses = predict_with_gemini(
        test_data['review_text'],
        api_key=api_key,
        model_name=os.getenv('GEMINI_MODEL', 'gemini-3.5-flash-lite'),
        temperature=0.0,  # dipakai otomatis untuk model Gemini versi lama
    )
else:
    gemini_predictions = [None] * len(test_data)
    gemini_raw_responses = [''] * len(test_data)
    print('Gemini belum dijalankan. Set GEMINI_API_KEY dan RUN_GEMINI=true/auto.')


## 5. Evaluasi dan perbandingan

In [ ]:
metrics_rows = [{'model': 'Classic ML', **classic_metrics}]
matrices = {'Classic ML': classic_matrix}

valid_gemini_predictions = [prediction is not None for prediction in gemini_predictions]
if any(valid_gemini_predictions):
    gemini_metrics, gemini_matrix, gemini_report = calculate_metrics(
        test_data['sentiment'], gemini_predictions
    )
    metrics_rows.append({'model': 'Gemini API', **gemini_metrics})
    matrices['Gemini API'] = gemini_matrix
else:
    gemini_metrics = None
    print('Metrik Gemini belum tersedia karena inference API belum dijalankan.')

comparison_df = pd.DataFrame(metrics_rows)
display(comparison_df.round(4))

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
predictions_output = test_data[['review_id', 'product_name', 'review_text', 'sentiment']].copy()
predictions_output['classic_prediction'] = classic_predictions
predictions_output['gemini_prediction'] = gemini_predictions
predictions_output['gemini_raw_response'] = gemini_raw_responses
predictions_output.to_csv(RESULTS_DIR / 'predictions.csv', index=False)
comparison_df.to_csv(RESULTS_DIR / 'model_comparison.csv', index=False)
save_confusion_matrices(matrices, DOCUMENTATION_DIR / 'model_comparison_summary.png')

print('File output tersimpan di folder results/ dan documentation/.')


In [ ]:
fig, axes = plt.subplots(1, len(matrices), figsize=(6 * len(matrices), 5))
if len(matrices) == 1:
    axes = [axes]

for axis, (model_name, matrix) in zip(axes, matrices.items()):
    matrix.plot(kind='bar', ax=axis, title=f'Confusion Matrix - {model_name}')
    axis.set_xlabel('Actual label')
    axis.set_ylabel('Jumlah')
    axis.tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()


## 6. Catatan interpretasi

- **Accuracy** menunjukkan proporsi seluruh prediksi yang benar.
- **Precision** menunjukkan seberapa sering prediksi positif memang benar positif.
- **Recall** menunjukkan seberapa banyak ulasan positif yang berhasil ditemukan.
- **F1-score** menyeimbangkan precision dan recall.

Kesimpulan akhir, analisis trade-off, limitation, dan rekomendasi technical approach didokumentasikan di `README.md` setelah eksperimen dijalankan.